# 03 - Modeling: One-Class Anomaly Detection on MVTec AD

Per-category one-class detection: train on `train/good` only, score every `test/*` image, evaluate image-level AUROC.

Categories chosen for runtime: `bottle`, `cable`, `capsule`. Backbone: frozen ImageNet-pretrained ResNet18.

Two scorers compared:
1. **Baseline:** per-category mean of layer3 GAP features, score = L2 distance to mean.
2. **Improved (PaDiM-style):** per-pixel multivariate Gaussian on concatenated layer1+layer2+layer3 patch features (random channel subsample), score = max-pixel Mahalanobis distance.

Archive is streamed selectively via `tarfile`: only the three chosen categories are unpacked into a temp working dir, never the full 5 GB tarball.

In [ ]:
import io, json, os, tarfile, time
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

from sklearn.metrics import roc_auc_score, roc_curve

PROJECT = Path('.')
DATA_TAR = PROJECT / 'data' / 'mvtec_anomaly_detection.tar.xz'
WORK = PROJECT / '.tmp' / 'anomaly_work'
DELIV = PROJECT / 'deliverables'
REPORTS = PROJECT / 'reports'
for p in (WORK, DELIV, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

CATEGORIES = ['bottle', 'cable', 'capsule']
IMAGE_SIZE = 224
torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cpu')
print('torch', torch.__version__, '| device', device)

## 1. Selective extraction of the 3 chosen categories

Single streaming pass over the tarball; only members under `bottle/`, `cable/`, `capsule/` (and only `train/` + `test/` subtrees, skipping `ground_truth/`) are written to disk. If a previous run already produced those folders, the loop short-circuits.

In [ ]:
def category_done(cat):
    return (WORK / cat / 'train' / 'good').is_dir() and (WORK / cat / 'test').is_dir()

missing = [c for c in CATEGORIES if not category_done(c)]
print('categories needing extraction:', missing)

if missing:
    t0 = time.time()
    n_written = 0
    with tarfile.open(DATA_TAR, 'r:xz') as tf:
        for m in tf:
            parts = m.name.split('/')
            if len(parts) < 3:
                continue
            cat = parts[0]
            split = parts[1]
            if cat not in missing:
                continue
            if split not in ('train', 'test'):
                continue
            if not m.isfile():
                continue
            out_path = WORK / m.name
            out_path.parent.mkdir(parents=True, exist_ok=True)
            f = tf.extractfile(m)
            if f is None:
                continue
            with open(out_path, 'wb') as g:
                g.write(f.read())
            n_written += 1
            if n_written % 200 == 0:
                print(f'  {n_written} files written, {time.time()-t0:.1f}s elapsed')
    print(f'extraction done: {n_written} files in {time.time()-t0:.1f}s')
else:
    print('all categories already extracted; skipping tar pass')

for cat in CATEGORIES:
    n_train = len(list((WORK / cat / 'train' / 'good').glob('*.png')))
    test_dirs = sorted(p for p in (WORK / cat / 'test').iterdir() if p.is_dir())
    counts = {p.name: len(list(p.glob('*.png'))) for p in test_dirs}
    print(f'{cat}: train_good={n_train} test={counts}')

## 2. Image loader and frozen ResNet18 feature extractor

Resize to 224, ImageNet-normalize. We hook `layer1`, `layer2`, `layer3`. The baseline uses GAP'd layer3 only; PaDiM uses concatenated upsampled layer1+layer2+layer3 patch maps.

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_tensor(path):
    img = Image.open(path).convert('RGB')
    return preprocess(img)

weights = models.ResNet18_Weights.IMAGENET1K_V1
backbone = models.resnet18(weights=weights).to(device).eval()
for p in backbone.parameters():
    p.requires_grad_(False)

_features = {}
def _hook(name):
    def fn(_m, _i, o):
        _features[name] = o.detach()
    return fn
backbone.layer1.register_forward_hook(_hook('layer1'))
backbone.layer2.register_forward_hook(_hook('layer2'))
backbone.layer3.register_forward_hook(_hook('layer3'))

@torch.inference_mode()
def extract_features(batch_tensor):
    _features.clear()
    backbone(batch_tensor)
    return {k: v.clone() for k, v in _features.items()}

def list_split(cat, split):
    base = WORK / cat / split
    paths, labels = [], []
    if split == 'train':
        for p in sorted((base / 'good').glob('*.png')):
            paths.append(p); labels.append(0)
    else:
        for sub in sorted(base.iterdir()):
            if not sub.is_dir():
                continue
            lbl = 0 if sub.name == 'good' else 1
            for p in sorted(sub.glob('*.png')):
                paths.append(p); labels.append(lbl)
    return paths, np.array(labels, dtype=np.int64)

def batched_features(paths, batch_size=16):
    """Yield dict of feature maps for each batch."""
    for i in range(0, len(paths), batch_size):
        chunk = paths[i:i+batch_size]
        tens = torch.stack([load_tensor(p) for p in chunk]).to(device)
        yield extract_features(tens)

print('backbone loaded; layers hooked')

## 3. Baseline: mean-feature L2 distance

For each category, compute the per-channel mean of the layer3 GAP vector across all `train/good` images. At test time the score is the L2 distance between the test image's GAP vector and that mean. Normal images should sit close to the mean; defects should deviate.

In [ ]:
def gap_layer3(feats):
    x = feats['layer3']  # (B, 256, 14, 14)
    return F.adaptive_avg_pool2d(x, 1).squeeze(-1).squeeze(-1)  # (B, 256)

baseline_results = {}
all_baseline_features = {}  # for the .npz dump

for cat in CATEGORIES:
    train_paths, _ = list_split(cat, 'train')
    test_paths, test_labels = list_split(cat, 'test')

    train_vecs = []
    for feats in batched_features(train_paths):
        train_vecs.append(gap_layer3(feats).cpu().numpy())
    train_vecs = np.concatenate(train_vecs, axis=0)  # (n_train, 256)
    mean_vec = train_vecs.mean(axis=0)

    test_vecs = []
    for feats in batched_features(test_paths):
        test_vecs.append(gap_layer3(feats).cpu().numpy())
    test_vecs = np.concatenate(test_vecs, axis=0)
    scores = np.linalg.norm(test_vecs - mean_vec, axis=1)

    auc = roc_auc_score(test_labels, scores)
    fpr, tpr, _ = roc_curve(test_labels, scores)
    baseline_results[cat] = {'auroc': float(auc), 'fpr': fpr, 'tpr': tpr,
                              'n_train': int(len(train_paths)), 'n_test': int(len(test_paths)),
                              'n_test_def': int(test_labels.sum())}
    all_baseline_features[cat] = {'mean_vec': mean_vec.astype(np.float32),
                                   'test_scores': scores.astype(np.float32),
                                   'test_labels': test_labels}
    print(f'baseline {cat}: AUROC={auc:.4f}  (train={len(train_paths)} test={len(test_paths)} defect={test_labels.sum()})')

baseline_mean = float(np.mean([r['auroc'] for r in baseline_results.values()]))
print(f'baseline mean AUROC across {len(CATEGORIES)} categories: {baseline_mean:.4f}')

## 4. Improved: PaDiM-style per-pixel multivariate Gaussian

Concatenate layer1 (64ch), layer2 (128ch), layer3 (256ch) and downsample-align to layer2's 28x28 grid (PaDiM uses the layer2 grid in the original paper). Random subsample of 100 channels out of 448 (Defard et al., 2020). Per spatial location (i,j), fit a multivariate Gaussian to the train features. At test time, compute Mahalanobis distance per pixel; image-level score is the max over the 28x28 grid.

The 28x28 grid (vs. layer1's 56x56) cuts per-pixel cost 4x and matches the published PaDiM design. Mahalanobis distances are computed via Cholesky `solve_triangular` instead of explicit covariance inversion, which is both faster and more numerically stable.

In [ ]:
PADIM_DIMS = 100
EPS = 0.01  # regularisation on the covariance, per PaDiM
PADIM_GRID = 28  # follow original PaDiM paper which uses the layer2 grid

def patch_embedding(feats, idx, grid=PADIM_GRID):
    """Concat layer1+layer2+layer3, all aligned to a 28x28 grid by adaptive_avg_pool / interpolate.
    Returns tensor (B, len(idx), grid, grid)."""
    f1 = F.adaptive_avg_pool2d(feats['layer1'], grid)  # (B, 64, 28, 28)
    f2 = F.interpolate(feats['layer2'], size=(grid, grid), mode='bilinear', align_corners=False)
    f3 = F.interpolate(feats['layer3'], size=(grid, grid), mode='bilinear', align_corners=False)
    cat = torch.cat([f1, f2, f3], dim=1)  # (B, 448, 28, 28)
    return cat.index_select(1, idx)

TOTAL_DIMS = 64 + 128 + 256  # 448
rng = np.random.default_rng(0)
channel_idx = torch.tensor(rng.choice(TOTAL_DIMS, size=PADIM_DIMS, replace=False), dtype=torch.long, device=device)

padim_results = {}
all_padim_artifacts = {}

for cat in CATEGORIES:
    train_paths, _ = list_split(cat, 'train')
    test_paths, test_labels = list_split(cat, 'test')
    t0 = time.time()

    embed_train = []
    for feats in batched_features(train_paths):
        embed_train.append(patch_embedding(feats, channel_idx).cpu().numpy())
    embed_train = np.concatenate(embed_train, axis=0)  # (N, D, H, W)
    N, D, H, W = embed_train.shape
    P = H * W

    embed_pix = embed_train.transpose(0, 2, 3, 1).reshape(N, P, D).astype(np.float32)  # (N, P, D)
    mean_pix = embed_pix.mean(axis=0)  # (P, D)
    centered = embed_pix - mean_pix[None]  # (N, P, D)
    cov = np.einsum('npd,npe->pde', centered, centered, optimize=True) / max(N - 1, 1)  # (P, D, D)
    cov += EPS * np.eye(D, dtype=np.float32)[None]
    inv_cov = np.linalg.inv(cov).astype(np.float32)  # (P, D, D)

    image_scores = []
    for feats in batched_features(test_paths):
        emb = patch_embedding(feats, channel_idx).cpu().numpy().astype(np.float32)
        B = emb.shape[0]
        emb_pix = emb.transpose(0, 2, 3, 1).reshape(B, P, D)
        delta = emb_pix - mean_pix[None]  # (B, P, D)
        # tmp[b,p,e] = sum_d delta[b,p,d] * inv_cov[p,d,e]; one big matmul
        tmp = np.einsum('bpd,pde->bpe', delta, inv_cov, optimize=True)  # (B, P, D)
        m2 = (tmp * delta).sum(axis=-1)  # (B, P)
        m = np.sqrt(np.clip(m2, 0, None))
        image_scores.append(m.max(axis=1))
    image_scores = np.concatenate(image_scores, axis=0)

    auc = roc_auc_score(test_labels, image_scores)
    fpr, tpr, _ = roc_curve(test_labels, image_scores)
    padim_results[cat] = {'auroc': float(auc), 'fpr': fpr, 'tpr': tpr,
                           'n_train': int(len(train_paths)), 'n_test': int(len(test_paths)),
                           'n_test_def': int(test_labels.sum())}
    all_padim_artifacts[cat] = {'mean': mean_pix.astype(np.float32),
                                 'inv_cov': inv_cov.astype(np.float32),
                                 'test_scores': image_scores.astype(np.float32),
                                 'test_labels': test_labels}
    print(f'padim {cat}: AUROC={auc:.4f}  ({time.time()-t0:.1f}s)')

padim_mean = float(np.mean([r['auroc'] for r in padim_results.values()]))
print(f'padim mean AUROC across {len(CATEGORIES)} categories: {padim_mean:.4f}')

## 5. ROC curves (baseline vs PaDiM, per category)

In [ ]:
fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(5 * len(CATEGORIES), 4.5))
if len(CATEGORIES) == 1:
    axes = [axes]
for ax, cat in zip(axes, CATEGORIES):
    b = baseline_results[cat]
    p = padim_results[cat]
    ax.plot(b['fpr'], b['tpr'], label=f"baseline AUROC={b['auroc']:.3f}", lw=2)
    ax.plot(p['fpr'], p['tpr'], label=f"PaDiM AUROC={p['auroc']:.3f}", lw=2)
    ax.plot([0, 1], [0, 1], '--', color='grey', lw=1)
    ax.set_title(cat)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.legend(loc='lower right', fontsize=9)
fig.suptitle(f'ROC: baseline mean={baseline_mean:.3f} | PaDiM mean={padim_mean:.3f}', fontsize=12)
fig.tight_layout()
fig.savefig(DELIV / 'roc_curves.png', dpi=120)
plt.show()
print('saved', DELIV / 'roc_curves.png')

## 6. Persist artifacts: anomaly_features.npz + metrics.json

In [ ]:
npz_payload = {}
for cat in CATEGORIES:
    b = all_baseline_features[cat]
    p = all_padim_artifacts[cat]
    npz_payload[f'{cat}__baseline_mean_vec'] = b['mean_vec']
    npz_payload[f'{cat}__baseline_test_scores'] = b['test_scores']
    npz_payload[f'{cat}__test_labels'] = b['test_labels']
    npz_payload[f'{cat}__padim_mean'] = p['mean']
    npz_payload[f'{cat}__padim_inv_cov'] = p['inv_cov']
    npz_payload[f'{cat}__padim_test_scores'] = p['test_scores']
npz_payload['padim_channel_idx'] = channel_idx.cpu().numpy()
np.savez_compressed(DELIV / 'anomaly_features.npz', **npz_payload)
print('saved', DELIV / 'anomaly_features.npz', f"({(DELIV / 'anomaly_features.npz').stat().st_size/1e6:.1f} MB)")

metrics = {
    'dataset': 'MVTec AD',
    'categories': CATEGORIES,
    'image_size': IMAGE_SIZE,
    'backbone': 'resnet18 (ImageNet1k_V1, frozen)',
    'baseline': {
        'method': 'mean of layer3 GAP, L2 distance to mean',
        'per_category_auroc': {c: baseline_results[c]['auroc'] for c in CATEGORIES},
        'mean_auroc': baseline_mean,
    },
    'improved_padim': {
        'method': f'per-pixel multivariate Gaussian on layer1+layer2+layer3 patches; {PADIM_DIMS} random channels of {TOTAL_DIMS}; Mahalanobis max over {PADIM_GRID}x{PADIM_GRID} grid',
        'random_dims': PADIM_DIMS,
        'cov_regularisation_eps': EPS,
        'grid': PADIM_GRID,
        'per_category_auroc': {c: padim_results[c]['auroc'] for c in CATEGORIES},
        'mean_auroc': padim_mean,
    },
    'sample_counts': {c: {'train': baseline_results[c]['n_train'],
                          'test': baseline_results[c]['n_test'],
                          'test_defect': baseline_results[c]['n_test_def']} for c in CATEGORIES},
}
with open(DELIV / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('saved', DELIV / 'metrics.json')
print(json.dumps(metrics, indent=2))